## 打印可视化界面

In [2]:
import pandas as pd

training_s = pd.read_csv(r'D:\20250416-LiHaiYu\models\CV-1\resnet50\viz/BST_TRAIN_RESULTS_SPEC.csv')
samples = list(training_s['fpath'])

In [ ]:
from onekey_algo.datasets.image_loader import default_loader
from onekey_algo.custom.components.comp2 import show_cam_on_image
import torch
import os
import random

from onekey_algo.custom.components.comp2 import extract, init_from_model, init_from_onekey
from onekey_algo.utils.MultiProcess import MultiProcess
from onekey_algo.custom.components.comp2 import target_layer_mapping
import os
import matplotlib
matplotlib.use('Agg')
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import monai
from glob import glob
import matplotlib.pyplot as plt
import pandas as pd

from glob import glob
# samples = glob(os.path.join(get_param_in_cwd('data pattern'), '*', '*.jpg')
# samples = glob(r'E:\20240419-WeiHangPing\Task2\patches/*/*.jpg')
model_dir = r'D:\20250416-LiHaiYu\models\CV-1\resnet50'
random.shuffle(samples)
def viz_sample(samples, thread_id):
    model, transformer, device = init_from_onekey(os.path.join(model_dir, 'viz'), 
                                                  force_pth=r'D:/20250416-LiHaiYu/models/CV-1/resnet50/train/training-params-1.pth')
#     for n, m in model.named_modules():
#         print('Feature name:', n, "|| Module:", m)
    target_layer = target_layer_mapping[os.path.basename(model_dir) + '_2D']
#     target_layer = 'features.denseblock4.denselayer16.conv2'
#     return
    gradcam = monai.visualize.GradCAM(nn_module=model, target_layers=target_layer)

    random.shuffle(samples)
    viz_dir = os.path.join(model_dir, 'Grad-CAM')
    os.makedirs(viz_dir, exist_ok=True)
    for sample in samples[:4096]:
        if not os.path.exists(sample):
            continue
        img = default_loader(sample)
        sample_ = transformer(img)
        sample_  = sample_.view(1, *sample_.size()).to(device)
        res_cam = gradcam(x=sample_, class_idx=None)
        fig, axes = plt.subplots(1, 2, figsize=(20, 10), facecolor='white')
    #     axes[0].imshow(-res_cam[0][0].cpu(), cmap='jet')
        axes[0].imshow(img.resize(sample_.size()[2:]))
        axes[0].axis('off')
    #     plt.savefig(f"viz/{os.path.basename(sample).replace('.png', '_se.png')}", bbox_inches = 'tight')
    #     plt.show()
    #     plt.figure(figsize=(10, 10))
    #     plt.axis('off')
        imshow = axes[1].imshow(show_cam_on_image(img.resize(sample_.size()[2:]), -res_cam[0][0].cpu(), use_rgb=True, reverse=False), 
                                cmap='jet')
        axes[1].axis('off')
        cax = fig.add_axes([0.92, 0.15, 0.02, axes[1].get_position().height]) 
        plt.colorbar(imshow, cax=cax)
        plt.savefig(f'{viz_dir}/{os.path.basename(sample)}', bbox_inches = 'tight')
        plt.close()
viz_sample(samples, thread_id=1)
# MultiProcess(func=viz_sample, samples=samples, num_process=1).run()

[2025-09-12 23:36:50 - comp2.py: 213]	INFO	Using force param files: D:/20250416-LiHaiYu/models/CV-1/resnet50/train/training-params-1.pth
[2025-09-12 23:36:50 - comp2.py: 234]	INFO	模型参数：{'pretrained': False, 'model_name': 'resnet50', 'num_classes': 2, 'in_channels': 3}
